In [ ]:
# ============================================================
# LOCAL OUTLIER FACTOR (LOF)
# ============================================================
#
# LEVEL: 🟡 MEDIUM
#
# Goal:
#   Understand LOF internally + use sklearn.
#
# We will NOT implement LOF from scratch.
#
# Topics:
#   1. Why LOF?
#   2. Intuition
#   3. Internal working
#   4. Important mathematics
#   5. Important parameters
#   6. sklearn implementation
#   7. Detect anomalies
#   8. Score anomalies
#   9. New/unseen data
#  10. LOF vs Isolation Forest
#  11. LOF vs DBSCAN
#  12. Interview questions
#  13. Working tree
#
# ============================================================


# ============================================================
# 1. WHY LOF?
# ============================================================

# LOF = Local Outlier Factor
#
# It is an UNSUPERVISED anomaly detection algorithm.
#
# Main question:
#
#     "Is this point much less dense than its neighbors?"
#
#
# Example:
#
#        ● ● ● ●
#       ● ● ● ● ●
#        ● ● ● ●
#
# This is a dense region.
#
#
#                    X
#
# X is far away from the nearby points.
#
# Its local density is much lower.
#
# Therefore:
#
#     X → possible anomaly
#
#
# ============================================================


# ============================================================
# 2. MAIN IDEA
# ============================================================

# LOF does NOT simply ask:
#
#     "Is this point far from the entire dataset?"
#
#
# Instead it asks:
#
#     "How dense is this point compared with
#      the density around its neighbors?"
#
#
# This is why LOF is called:
#
#     LOCAL Outlier Factor
#
#
# Important:
#
#     Isolation Forest
#         → isolation/path length
#
#     LOF
#         → local density
#
#
# ============================================================


# ============================================================
# 3. WHY LOCAL DENSITY?
# ============================================================

# Consider two clusters:
#
#
# Dense cluster:
#
#       ● ● ● ●
#      ● ● ● ● ●
#       ● ● ● ●
#
#
# Sparse cluster:
#
#                 ●     ●
#                    ●
#              ●          ●
#
#
# A point in the sparse cluster may have lower density
# than points in the dense cluster.
#
# But that does NOT automatically make it an anomaly.
#
# It should be compared with its LOCAL neighbors.
#
#
# If:
#
#     density(point) ≈ density(neighbors)
#
#         → probably normal
#
#
# If:
#
#     density(point) << density(neighbors)
#
#         → potential anomaly
#
#
# ============================================================


# ============================================================
# 4. INTERNAL WORKING
# ============================================================

# For every point P:
#
#
# STEP 1
#     Find its k nearest neighbors.
#
# STEP 2
#     Calculate k-distance.
#
# STEP 3
#     Calculate reachability distance.
#
# STEP 4
#     Calculate local reachability density.
#
# STEP 5
#     Compare P's density with neighbor densities.
#
# STEP 6
#     Calculate LOF score.
#
#
# Overall:
#
#
#        Point P
#           |
#           v
#    Find k nearest neighbors
#           |
#           v
#       k-distance
#           |
#           v
#  Reachability distances
#           |
#           v
# Local reachability density
#           |
#           v
# Compare with neighbors
#           |
#           v
#       LOF score
#
#
# ============================================================


# ============================================================
# 5. STEP 1 — k NEAREST NEIGHBORS
# ============================================================

# Choose:
#
#     k = n_neighbors
#
#
# Example:
#
#     k = 5
#
# For every point, find its 5 nearest neighbors.
#
#
# Small k:
#
#     very local
#     can detect small/local anomalies
#     more sensitive to noise
#
#
# Large k:
#
#     broader neighborhood
#     smoother
#     may miss very local anomalies
#
#
# IMPORTANT PARAMETER:
#
#     n_neighbors
#
#
# ============================================================


# ============================================================
# 6. STEP 2 — k-DISTANCE
# ============================================================

# Suppose:
#
#     k = 3
#
# For point P:
#
#     1st nearest neighbor = distance 2
#     2nd nearest neighbor = distance 3
#     3rd nearest neighbor = distance 5
#
#
# Therefore:
#
#     k-distance(P) = 5
#
#
# This gives us a measure of the local scale around P.
#
#
# ============================================================


# ============================================================
# 7. STEP 3 — REACHABILITY DISTANCE
# ============================================================

# LOF uses:
#
#     reachability distance
#
#
# Formula:
#
#
# reach-dist_k(P, O)
#
#     = max(
#           k-distance(O),
#           distance(P, O)
#       )
#
#
# where:
#
#     P = current point
#     O = neighboring point
#
#
# Why?
#
# It prevents very small distances in extremely dense
# regions from making density calculations unstable.
#
#
# ============================================================


# ============================================================
# 8. STEP 4 — LOCAL REACHABILITY DENSITY
# ============================================================

# Local reachability density:
#
#
#                    k
#                   ----
#                   \
#                    reach-dist(P, O)
#                   /
#                   ----
#
# LRD(P) ≈ -------------------------------
#                         k
#
#
# In simple words:
#
#     LRD ≈ inverse of average reachability distance
#
#
# Therefore:
#
#     small distances
#          ↓
#     high density
#
#
#     large distances
#          ↓
#     low density
#
#
# ============================================================


# ============================================================
# 9. STEP 5 — LOF SCORE
# ============================================================

# LOF compares:
#
#     neighbor density
#
# against:
#
#     point density
#
#
# Simplified:
#
#
#                  average LRD of neighbors
# LOF(P) ≈ -----------------------------------
#                       LRD(P)
#
#
# Therefore:
#
#
# LOF ≈ 1
#     → similar density to neighbors
#     → probably normal
#
#
# LOF > 1
#     → lower density than neighbors
#     → possible anomaly
#
#
# LOF >> 1
#     → much lower density
#     → stronger anomaly signal
#
#
# ============================================================


# ============================================================
# 10. SMALL NUMERICAL EXAMPLE
# ============================================================

# Suppose:
#
#     LRD(P) = 0.20
#
#
# Neighbor densities:
#
#     0.80
#     0.90
#     0.70
#     0.80
#
#
# Average neighbor density:
#
#     (0.80 + 0.90 + 0.70 + 0.80) / 4
#
#     = 0.80
#
#
# LOF:
#
#     0.80 / 0.20
#
#     = 4
#
#
# LOF ≈ 4
#
# This means P is much less dense than its neighbors.
#
# Therefore P is a strong anomaly candidate.
#
#
# ============================================================


# ============================================================
# 11. IMPORTANT PARAMETERS
# ============================================================

# n_neighbors:
#
#     Size of local neighborhood.
#
#     Most important parameter.
#
#
# contamination:
#
#     Expected proportion of outliers.
#
#     Example:
#
#         contamination=0.05
#
#     approximately 5% are expected to be anomalies.
#
#
# metric:
#
#     Distance metric.
#
#     Examples:
#
#         "euclidean"
#         "manhattan"
#         "minkowski"
#         "cosine"
#
#
# novelty:
#
#     False:
#         Detect outliers in the training dataset.
#
#     True:
#         Allows scoring/predicting NEW unseen observations.
#
#
# n_jobs:
#
#     Number of CPU workers for neighbor search.
#
#
# leaf_size:
#
#     Controls tree-based nearest-neighbor search.
#
#
# ============================================================


# ============================================================
# 12. INSTALLATION
# ============================================================

# sklearn is normally already available.
#
# If needed:
#
# %pip install scikit-learn
#
#
# ============================================================


# ============================================================
# 13. IMPORTS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import LocalOutlierFactor


# ============================================================
# 14. CREATE SIMPLE DATASET
# ============================================================

# Normal observations:
#
# Dense cluster around (0, 0)
#
rng = np.random.RandomState(42)

X_normal = 0.5 * rng.randn(300, 2)


# Anomalies:
#
# Far away from the normal cluster.
#
X_anomaly = rng.uniform(
    low=4,
    high=6,
    size=(15, 2)
)


# Combine them.
#
X = np.vstack([
    X_normal,
    X_anomaly
])

print("Dataset shape:", X.shape)


# ============================================================
# 15. TRAIN LOF
# ============================================================

# n_neighbors:
#     Examine approximately 20 nearby points.
#
# contamination:
#     Expect around 5% anomalies.
#
# metric:
#     Euclidean distance.
#
# n_jobs:
#     Use all available CPU cores.
#

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    metric="euclidean",
    n_jobs=-1
)


# fit_predict:
#
#     Fits LOF and immediately returns:
#
#     +1 → normal
#     -1 → anomaly
#
predictions = lof.fit_predict(X)


print("Unique predictions:", np.unique(predictions))

print(
    "Number of anomalies:",
    np.sum(predictions == -1)
)


# ============================================================
# 16. VISUALIZE
# ============================================================

plt.figure(figsize=(9, 7))

# Normal points
plt.scatter(
    X[predictions == 1, 0],
    X[predictions == 1, 1],
    label="Normal",
    s=20
)

# Anomalies
plt.scatter(
    X[predictions == -1, 0],
    X[predictions == -1, 1],
    label="Anomaly",
    s=50,
    marker="x"
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Local Outlier Factor")
plt.legend()

plt.show()


# ============================================================
# 17. GET LOF SCORES
# ============================================================

# sklearn exposes:
#
#     negative_outlier_factor_
#
#
# Important:
#
# sklearn stores the score with the opposite sign from
# the intuitive positive LOF interpretation.
#
#
# Therefore:
#
#     intuitive_LOF ≈ -negative_outlier_factor_
#
#
# Higher intuitive LOF:
#
#     more anomalous
#

lof_scores = -lof.negative_outlier_factor_

print("First 10 LOF scores:")
print(lof_scores[:10])


# ============================================================
# 18. FIND MOST ANOMALOUS POINTS
# ============================================================

# Higher LOF = more anomalous.
#
most_anomalous_indices = np.argsort(
    lof_scores
)[-10:][::-1]


print("Most anomalous points:")

print(
    X[most_anomalous_indices]
)


print("Their LOF scores:")

print(
    lof_scores[most_anomalous_indices]
)


# ============================================================
# 19. IMPORTANT — novelty=False vs novelty=True
# ============================================================

# By default:
#
#     novelty=False
#
#
# This is mainly for detecting outliers in the dataset
# used during fit.
#
#
# Example:
#
#     predictions = lof.fit_predict(X)
#
#
# If you want to fit on training data and later detect
# anomalies in NEW unseen data:
#
#     novelty=True
#
#
# ============================================================
# 20. LOF FOR NEW UNSEEN DATA
# ============================================================

lof_new = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    metric="euclidean",
    novelty=True,
    n_jobs=-1
)


# Fit only on training/normal data.
#
lof_new.fit(X)


# ============================================================
# 21. CREATE NEW OBSERVATIONS
# ============================================================

new_points = np.array([
    [0.1, 0.2],     # likely normal
    [0.5, -0.2],    # likely normal
    [4.5, 4.5],     # likely anomaly
    [5.5, 5.0]      # likely anomaly
])


# Predict:
#
#     +1 → normal
#     -1 → anomaly
#

new_predictions = lof_new.predict(new_points)

print("New predictions:")
print(new_predictions)


# ============================================================
# 22. SCORE NEW DATA
# ============================================================

# decision_function:
#
#     positive → more normal
#     negative → more anomalous
#

decision_scores = lof_new.decision_function(
    new_points
)

print("Decision scores:")
print(decision_scores)


# ============================================================
# 23. SCORE_SAMPLES
# ============================================================

# score_samples gives an anomaly-related score.
#
# Lower / more negative:
#
#     more abnormal
#

sample_scores = lof_new.score_samples(
    new_points
)

print("Sample scores:")
print(sample_scores)


# ============================================================
# 24. FEATURE SCALING
# ============================================================

# LOF depends on distances.
#
# Therefore, feature scale matters.
#
#
# Example:
#
#     age       → 20 to 60
#     salary    → 20,000 to 2,000,000
#
#
# Salary can dominate Euclidean distance.
#
#
# In such cases use StandardScaler.
#

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# Train LOF on scaled data.
#
lof_scaled = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    metric="euclidean",
    n_jobs=-1
)

pred_scaled = lof_scaled.fit_predict(
    X_scaled
)

print(
    "Anomalies after scaling:",
    np.sum(pred_scaled == -1)
)


# ============================================================
# 25. HIGH-DIMENSIONAL DATA
# ============================================================

# LOF relies heavily on distances.
#
# In very high dimensions:
#
#     distances become less informative.
#
# This is called the:
#
#     Curse of Dimensionality
#
#
# Possible approach:
#
#
#     High-dimensional X
#           |
#           v
#          PCA
#           |
#           v
#     Lower-dimensional X
#           |
#           v
#          LOF
#
#
# But this should be validated for the specific problem.
#
#
# ============================================================
# 26. LOF vs ISOLATION FOREST
# ============================================================

# Isolation Forest:
#
#     Main question:
#
#         "How quickly can I isolate this point?"
#
#
# Mechanism:
#
#     Random trees
#
# Signal:
#
#     Path length
#
#
# ------------------------------------------------------------
#
#
# LOF:
#
#     Main question:
#
#         "Is this point much less dense than its neighbors?"
#
#
# Mechanism:
#
#     Nearest neighbors
#
# Signal:
#
#     Relative local density
#
#
# ============================================================
# 27. LOF vs DBSCAN
# ============================================================

# DBSCAN:
#
#     Finds dense clusters.
#
#     Main parameters:
#
#         eps
#         min_samples
#
#
# LOF:
#
#     Measures whether a point has unusually low local density.
#
#     Main parameter:
#
#         n_neighbors
#
#
# DBSCAN asks:
#
#     "Is this point part of a dense region?"
#
#
# LOF asks:
#
#     "Is this point less dense than its neighbors?"
#
#
# ============================================================
# 28. LOF vs KNN
# ============================================================

# Both use nearest neighbors.
#
#
# KNN:
#
#     Usually supervised.
#
#     Uses labels to make predictions.
#
#
# LOF:
#
#     Unsupervised.
#
#     Uses distances to estimate local density.
#
#
# KNN:
#
#     "What do my neighbors tell me?"
#
#
# LOF:
#
#     "How does my density compare with my neighbors?"
#
#
# ============================================================
# 29. LOF vs ONE-CLASS SVM
# ============================================================

# LOF:
#
#     Neighbor/density based.
#
#
# One-Class SVM:
#
#     Learns a boundary around normal observations.
#
#
# LOF:
#
#     Local view.
#
#
# One-Class SVM:
#
#     Boundary-based view.
#
#
# ============================================================
# 30. ADVANTAGES
# ============================================================

# 1. Detects local anomalies.
#
# 2. Does not require anomaly labels.
#
# 3. Can handle different local densities better than
#    purely global methods.
#
# 4. Supports different distance metrics.
#
# 5. Easy to use with sklearn.
#
#
# ============================================================
# 31. LIMITATIONS
# ============================================================

# 1. Can be computationally expensive on large datasets.
#
# 2. Sensitive to n_neighbors.
#
# 3. Sensitive to distance metric.
#
# 4. High-dimensional data can make distances less useful.
#
# 5. Feature scaling can strongly affect results.
#
# 6. contamination/threshold selection can be difficult.
#
#
# IMPORTANT FOR YOUR ML ENGINEERING WORK:
#
# For very large datasets, don't blindly run LOF on
# hundreds of thousands/millions of high-dimensional rows.
#
# Isolation Forest is often a more practical first choice.
#
#
# ============================================================
# 32. INTERVIEW QUESTIONS
# ============================================================

# Q1. What is LOF?
#
# Answer:
#
# LOF is an unsupervised, density-based anomaly detection
# algorithm that compares the local density of a point
# with the density of its nearest neighbors.
#
#
# Q2. Why is it called Local Outlier Factor?
#
# Answer:
#
# Because it measures how much a point's local density
# differs from the density of its local neighborhood.
#
#
# Q3. What does LOF ≈ 1 mean?
#
# Answer:
#
# The point has approximately the same density as its
# neighbors and is generally considered normal.
#
#
# Q4. What does LOF > 1 mean?
#
# Answer:
#
# The point is less dense than its neighbors and may be
# an outlier. Larger values indicate a stronger anomaly
# signal.
#
#
# Q5. What is n_neighbors?
#
# Answer:
#
# It controls the size of the local neighborhood used
# to estimate density.
#
#
# Q6. Why is scaling important?
#
# Answer:
#
# LOF is distance-based, so features with larger numerical
# scales can dominate the distance calculation.
#
#
# Q7. LOF vs Isolation Forest?
#
# Answer:
#
# Isolation Forest uses random tree partitions and
# isolation path length, whereas LOF uses nearest
# neighbors and relative local density.
#
#
# Q8. What is novelty=True?
#
# Answer:
#
# It allows the fitted LOF model to score and predict
# anomalies for new unseen observations.
#
#
# ============================================================
# 33. 30-SECOND INTERVIEW ANSWER
# ============================================================

# "LOF is an unsupervised density-based anomaly detection
# algorithm. It finds the nearest neighbors of each point,
# estimates its local reachability density, and compares that
# density with the densities of its neighbors. If a point is
# significantly less dense than its neighborhood, its LOF
# score becomes larger and it is considered a potential
# anomaly. The most important parameter is n_neighbors because
# it determines the size of the local neighborhood."
#
#
# ============================================================
# 34. WORKING TREE
# ============================================================

#                       INPUT DATA
#                           |
#                           v
#                  Choose n_neighbors
#                           |
#                           v
#                Find nearest neighbors
#                           |
#                           v
#                     k-distance
#                           |
#                           v
#              Reachability distance
#                           |
#                           v
#          Local Reachability Density
#                           |
#                           v
#       Compare point density with neighbors
#                           |
#                           v
#                      LOF score
#                           |
#                -------------------
#                |                 |
#                v                 v
#             ≈ 1              >> 1
#                |                 |
#                v                 v
#             NORMAL            ANOMALY
#
#
# ============================================================
# 35. FINAL MENTAL MODEL
# ============================================================

# Isolation Forest:
#
#     "How quickly can I isolate this point?"
#
#
# LOF:
#
#     "Is this point much less dense than the points
#      around it?"
#
#
# DBSCAN:
#
#     "Is this point part of a dense region?"
#
#
# KNN:
#
#     "What do my nearest neighbors tell me?"
#
#
# ============================================================
# END OF LOF
# ============================================================